<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/hybrid-rag-poc/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução Estratégica
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 4 e/ou 5 e/ou 6.

In [ ]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/pln/hybrid-rag-poc'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

In [ ]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

In [ ]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

LIVRO_ID_ALVO = 18 # Livro de Jó, para PoC

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

In [ ]:
# Célula 3.1: Vinculação dos Versículos às Unidades Literárias
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM unidade_literaria WHERE livro_id = ?", (LIVRO_ID_ALVO,))
if cursor.fetchone()[0] == 0:
    print(f"ℹ️ Nenhuma unidade literária curada para livro_id={LIVRO_ID_ALVO}. Gerando fallback por capítulo...")
    cursor.execute("""
        INSERT INTO unidade_literaria (
            livro_id, capitulo_inicio, verso_inicio, capitulo_fim, verso_fim, tipo, interlocutor
        )
        SELECT livro_id, numero_capitulo, MIN(numero_verso), numero_capitulo, MAX(numero_verso),
               'capitulo_padrao', NULL
        FROM verso
        WHERE livro_id = ?
        GROUP BY livro_id, numero_capitulo
    """, (LIVRO_ID_ALVO,))
    conn.commit()
    print(f"✅ Fallback gerado: uma unidade por capítulo para livro_id={LIVRO_ID_ALVO}.")
else:
    print(f"ℹ️ livro_id={LIVRO_ID_ALVO} já possui unidades literárias curadas — fallback não aplicado.")

df_unidades = pd.read_sql_query(
    "SELECT * FROM unidade_literaria WHERE livro_id = ?", conn, params=(LIVRO_ID_ALVO,)
)
df_versos_ref = pd.read_sql_query(
    "SELECT id, numero_capitulo, numero_verso FROM verso WHERE livro_id = ?",
    conn, params=(LIVRO_ID_ALVO,)
)

def referencia_para_chave(cap, verso):
    return cap * 1000 + verso  # funciona bem se nenhum capítulo tiver >999 versículos

df_unidades['chave_inicio'] = df_unidades.apply(lambda r: referencia_para_chave(r['capitulo_inicio'], r['verso_inicio']), axis=1)
df_unidades['chave_fim'] = df_unidades.apply(lambda r: referencia_para_chave(r['capitulo_fim'], r['verso_fim']), axis=1)
df_versos_ref['chave'] = df_versos_ref.apply(lambda r: referencia_para_chave(r['numero_capitulo'], r['numero_verso']), axis=1)

def encontrar_unidade(chave):
    match = df_unidades[(df_unidades['chave_inicio'] <= chave) & (df_unidades['chave_fim'] >= chave)]
    return int(match.iloc[0]['id']) if not match.empty else None

df_versos_ref['unidade_literaria_id'] = df_versos_ref['chave'].apply(encontrar_unidade)

sem_unidade = df_versos_ref['unidade_literaria_id'].isna().sum()
if sem_unidade > 0:
    print(f"⚠️ {sem_unidade} versículo(s) não caíram em nenhuma unidade literária — revise os intervalos.")

cursor = conn.cursor()
cursor.executemany(
    "UPDATE verso SET unidade_literaria_id = ? WHERE id = ?",
    [(int(row['unidade_literaria_id']), int(row['id'])) for _, row in df_versos_ref.dropna(subset=['unidade_literaria_id']).iterrows()]
)
conn.commit()
conn.close()
print(f"✅ {len(df_versos_ref) - sem_unidade} versículos vinculados às suas unidades literárias.")

In [ ]:
# Célula 3.2: Geracao do banco vetorial chromadb, a partir de dados contidos em tabela de chunks no sqlite
# Install chromadb if not already installed
!pip install -q chromadb

import sqlite3
import os
import chromadb
from sentence_transformers import SentenceTransformer
from google.colab import drive

# 2. Caminhos dos arquivos e diretórios
base_dir = "/content/drive/MyDrive/pln/hybrid-rag-poc/data"
sqlite_db_path = os.path.join(base_dir, "base-dados.db")
chroma_persist_dir = os.path.join(base_dir, "chroma_db_moody")

print("🔄 Conectando ao SQLite e carregando os chunks...")
conn = sqlite3.connect(sqlite_db_path)
cursor = conn.cursor()

# Selecionar todos os campos necessários da tabela existente
cursor.execute("""
    SELECT
        chunk_id,
        livro_id,
        capitulo_inicio,
        verso_inicio,
        capitulo_fim,
        verso_fim,
        secao_n1,
        secao_n2,
        secao_n3,
        secao_n4,
        texto,
        pagina_origem
    FROM chunks_comentario;
""")
rows = cursor.fetchall()
conn.close()

print(f"📦 Total de {len(rows)} chunks carregados do SQLite para vetorização.")

if len(rows) == 0:
    print("⚠️ Atenção: Nenhum registro encontrado na tabela 'chunks_comentario' do SQLite.")
else:
    # 3. Inicializar o modelo de Embeddings Multilíngue
    print("🧠 Carregando modelo de embeddings (SentenceTransformers)...")
    embedding_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

    # 4. Inicializar o ChromaDB de forma persistente no Drive
    print(f"🗄️ Inicializando ChromaDB em: {chroma_persist_dir}")
    chroma_client = chromadb.PersistentClient(path=chroma_persist_dir)

    collection_name = "comentario_moody_jo"

    # Criar ou obter a coleção de forma limpa
    try:
        chroma_client.delete_collection(name=collection_name)
        print("🗑️ Coleção antiga removida para atualização limpa.")
    except Exception:
        pass

    NOME_MODELO_EMBEDDING_INGESTAO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={
            "description": "Comentário Bíblico Moody - Livro de Jó com metadados estruturados",
            "embedding_model": NOME_MODELO_EMBEDDING_INGESTAO,
            "embedding_dim": 384,
            "hnsw:space": "cosine"  # ESSENCIAL: garante que 'distance' seja distância de cosseno,
                                    # tornando melhor_score = 1 - distance matematicamente válido
        }
    )

    # 5. Preparar documentos, metadados e IDs
    docs = []
    metadatas = []
    ids = []

    for row in rows:
        (
            chunk_id, livro_id, capitulo_inicio, verso_inicio,
            capitulo_fim, verso_fim, secao_n1, secao_n2,
            secao_n3, secao_n4, texto, pagina_origem
        ) = row

        docs.append(str(texto))

        # Metadados normalizados rigorosamente tipados para os filtros do RAG
        metadatas.append({
            "livro_id": int(livro_id) if livro_id is not None else 18,
            "capitulo_inicio": int(capitulo_inicio) if capitulo_inicio is not None else 0,
            "verso_inicio": int(verso_inicio) if verso_inicio is not None else 0,
            "capitulo_fim": int(capitulo_fim) if capitulo_fim is not None else 0,
            "verso_fim": int(verso_fim) if verso_fim is not None else 0,
            "secao_n1": str(secao_n1) if secao_n1 is not None else "",
            "secao_n2": str(secao_n2) if secao_n2 is not None else "",
            "secao_n3": str(secao_n3) if secao_n3 is not None else "",
            "secao_n4": str(secao_n4) if secao_n4 is not None else "",
            "pagina_origem": int(pagina_origem) if pagina_origem is not None else 0
        })

        ids.append(str(chunk_id))

    print("⚡ Gerando embeddings e inserindo no ChromaDB...")
    embeddings = embedding_model.encode(docs, show_progress_bar=True).tolist()

    collection.add(
        documents=docs,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )

    print(f"✨ Indexação concluída com sucesso! Total de vetores armazenados na coleção '{collection_name}': {collection.count()}")

In [ ]:
# Célula 3.2: Inicialização e Controle de Execução do Pipeline (Governança)
import sqlite3
from datetime import datetime
import os

# 1. Configuração e Conexão com o Banco de Dados
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# 2. Identifica a versão ativa do framework
cursor.execute("SELECT id, codigo_versao FROM framework_versao ORDER BY id DESC LIMIT 1")
fw_row = cursor.fetchone()
if not fw_row:
    raise ValueError("❌ Nenhuma versão de framework encontrada. Execute a carga de seed SQL (Célula 3) primeiro.")

framework_versao_id, codigo_versao = fw_row

# 3. Registra a nova execução
data_execucao_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

cursor.execute("""
    INSERT INTO execucao_pipeline (
        data_execucao, framework_versao_id, observacoes, livro_id_alvo
    ) VALUES (?, ?, ?, ?)
""", (data_execucao_str, framework_versao_id, 'Nova execução de pipeline iniciada.', LIVRO_ID_ALVO))

execucao_id = cursor.lastrowid
conn.commit()
conn.close()

print(f"🚀 Nova Execução iniciada com sucesso!")
print(f"📌 Execução ID: {execucao_id} | Framework Vinculado: v{codigo_versao}")
print("O pipeline agora está pronto para rodar as Células 4, 5, 5.1 e 6 atreladas a este ID.")

In [ ]:
# Célula 4: Limpeza Estrutural e Filtro de Densidade (Antídoto ao Ruído Nominal)
import spacy
import sqlite3
import pandas as pd

# Carrega o modelo de português
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    get_ipython().system('python -m spacy download pt_core_news_lg -q')
    nlp = spacy.load("pt_core_news_lg")

def limpar_texto_estrutural(texto):
    if not texto or len(texto.strip()) < 3: return "RUIDO_CURTO"
    doc = nlp(texto)

    tokens = [t for t in doc if not t.is_stop and not t.is_punct and t.pos_ in ['NOUN', 'VERB', 'ADJ', 'PROPN']]
    if not tokens: return "RUIDO_VAZIO"

    propn_count = len([t for t in tokens if t.pos_ == 'PROPN'])
    propn_ratio = propn_count / len(tokens)
    has_action_or_state = any(t.pos_ in ['VERB', 'ADJ'] for t in tokens)

    # CASO CRÍTICO (Ex: Nesias e Hatifa)
    if len(tokens) <= 3 and propn_ratio > 0.5 and not has_action_or_state:
        return "RUIDO_NOMINAL"

    # FILTRO DINÂMICO PARA TEXTOS CURTOS
    if len(tokens) < 5:
        return " ".join([t.text.lower() for t in doc if not t.is_punct])

    return " ".join([t.text.lower() for t in tokens])

# 1. Conexão e Recuperação da Execução Vigente
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

if 'execucao_id' not in globals():
    cursor.execute("SELECT id FROM execucao_pipeline ORDER BY id DESC LIMIT 1")
    row = cursor.fetchone()
    if row:
        execucao_id = row[0]
        print(f"🔄 Retomando processamento atrelado à Execução ID: {execucao_id} (Recuperada do banco)")
    else:
        raise ValueError("❌ Nenhuma execução encontrada. Rode a Célula 3.1 de Inicialização primeiro.")
else:
    print(f"▶️ Processando dados para a Execução ID: {execucao_id}")

# 2. Processamento
df_versos = pd.read_sql_query("SELECT id, texto FROM verso WHERE livro_id = ?", conn, params=(LIVRO_ID_ALVO,))
print("🧼 Limpando textos e aplicando filtros de densidade gramatical...")
df_versos['texto_limpo'] = df_versos['texto'].apply(limpar_texto_estrutural)

# 3. Persistência Idempotente (Chave Composta)
print("💾 Persistindo na tabela verso_limpo (Upsert para reexecução segura)...")
sql_verso_limpo = """
    INSERT OR REPLACE INTO verso_limpo (verso_id, execucao_id, texto_limpo)
    VALUES (?, ?, ?)
"""
dados_limpos = [
    (row['id'], execucao_id, row['texto_limpo'])
    for _, row in df_versos.iterrows()
]

cursor.executemany(sql_verso_limpo, dados_limpos)
conn.commit()
conn.close()

print("✅ Célula 4 concluída! Ruídos nominais e estruturais pré-identificados.")

In [ ]:
# Célula 5: Classificação por Eixos Existenciais (Com Incerteza Estatística e Chaves Compostas)
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import sqlite3
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# 1. Configuração e Conexão
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# --- GOVERNANÇA: RECUPERAR EXECUÇÃO E ATUALIZAR MODELO DE TÓPICOS ---
if 'execucao_id' not in globals():
    cursor.execute("SELECT id, framework_versao_id FROM execucao_pipeline ORDER BY id DESC LIMIT 1")
    row = cursor.fetchone()
    if row:
        execucao_id, framework_versao_id = row
    else:
        raise ValueError("❌ Nenhuma execução encontrada. Rode a Célula 3.1 primeiro.")
else:
    cursor.execute("SELECT framework_versao_id FROM execucao_pipeline WHERE id = ?", (execucao_id,))
    framework_versao_id = cursor.fetchone()[0]

# NOME_MODELO_EMBEDDING = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
NOME_MODELO_EMBEDDING = "rufimelo/bert-large-portuguese-cased-sts"

# Atualiza a execução indicando qual modelo está sendo usado no BERTopic
cursor.execute("""
    UPDATE execucao_pipeline
    SET modelo_embeddings_topico = ?
    WHERE id = ?
""", (NOME_MODELO_EMBEDDING, execucao_id))
conn.commit()
print(f"📌 Execução {execucao_id} atualizada: Modelo de Tópicos definido.")

# --- CARREGAR TÓPICOS DA VERSÃO ATIVA DO FRAMEWORK (SSOT) ---
df_topicos = pd.read_sql_query(
    "SELECT id, nome_curto, descricao_ancora_zeroshot FROM topico WHERE framework_versao_id = ? ORDER BY id",
    conn, params=(framework_versao_id,)
)
descricoes_eixos = df_topicos['descricao_ancora_zeroshot'].tolist()
topico_ids_db = df_topicos['id'].tolist()

# 2. Carga de Dados (Garantindo que pegamos o texto limpo da MESMA execução)
query_versos = """
    SELECT v.id as verso_id, v.texto, v.unidade_literaria_id, l.abreviacao, g.id as genero_id, vl.texto_limpo
    FROM verso v
    JOIN verso_limpo vl ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario g ON g.id = l.genero_id
    WHERE v.livro_id = ? AND vl.execucao_id = ?
    ORDER BY v.id
"""
df_input = pd.read_sql_query(query_versos, conn, params=(LIVRO_ID_ALVO, execucao_id))
df_input['texto'] = df_input['texto'].fillna('').astype(str)
df_input['texto_limpo'] = df_input['texto_limpo'].fillna('vazio').astype(str)

# --- JANELAMENTO RESPEITANDO OS LIMITES DA UNIDADE LITERÁRIA ---
df_input['grupo_janela'] = df_input['unidade_literaria_id'].fillna(-df_input['verso_id'])
df_input['texto_anterior'] = df_input.groupby('grupo_janela')['texto'].shift(1, fill_value='')
df_input['texto_posterior'] = df_input.groupby('grupo_janela')['texto'].shift(-1, fill_value='')
df_input['texto_janela'] = (df_input['texto_anterior'] + " " +
                            df_input['texto'] + " " +
                            df_input['texto_posterior']).str.strip()

docs_para_classificar = []
for idx, row in df_input.iterrows():
    if row['texto_limpo'].startswith('RUIDO'):
        docs_para_classificar.append("vazio")
    else:
        docs_para_classificar.append(row['texto_janela'])

# 3. Inicialização do Modelo BERTopic
print(f"🔄 Carregando modelo unificado: {NOME_MODELO_EMBEDDING}")
embedding_model = SentenceTransformer(NOME_MODELO_EMBEDDING)

model_topic = BERTopic(
    embedding_model=embedding_model,
    zeroshot_topic_list=descricoes_eixos,
    zeroshot_min_similarity=0.1,
    calculate_probabilities=True,
    vectorizer_model=CountVectorizer(ngram_range=(1, 2))
)

print("🤖 Classificando via Zero-Shot...")
topics, probs_matrix = model_topic.fit_transform(docs_para_classificar)

# --- CORREÇÃO: mapear corretamente as colunas de probs_matrix para os topico_id reais ---
# O BERTopic ordena as colunas de probs_matrix na mesma ordem de get_topic_info()['Topic'],
# que inclui a coluna do outlier (-1) quando ele existe. Ler por posição (enumerate) assume
# erroneamente que a coluna N corresponde ao topico_id N, sem considerar esse deslocamento.
# --- CORREÇÃO ROBUSTA DA ORDEM DE COLUNAS DO BERTOPIC (AÇÃO 1) ---
ordem_colunas_topico = model_topic.get_topic_info()['Topic'].tolist()
print(f"🔎 Ordem real das colunas em probs_matrix: {ordem_colunas_topico}")

# Mapeia apenas os índices das colunas que correspondem aos IDs reais dos tópicos cadastrados no banco,
# ignorando explicitamente o tópico outlier (-1) se ele estiver presente.
coluna_por_topico_id = {}
for idx, topic_id in enumerate(ordem_colunas_topico):
    if topic_id in topico_ids_db:
        coluna_por_topico_id[topic_id] = idx

print(f"📌 Mapeamento seguro validado (Topic ID -> Índice da Matriz): {coluna_por_topico_id}")

# --- CALIBRAÇÃO: softmax com temperatura para medir dominância real entre os 4 scores brutos ---
# Os scores de similaridade de cosseno do BERTopic zero-shot NÃO formam uma distribuição
# categórica — são 4 medidas independentes. Normalizar por soma sub-representa vantagens
# moderadas (ex.: 0.55 vs 0.44) como "quase empate", inflando a entropia artificialmente.
# O softmax com temperatura converte os scores brutos em uma distribuição que amplifica
# a diferença relativa entre eles de forma controlada e ajustável.
TEMPERATURA_SOFTMAX = 0.15  # menor = mais sensível a pequenas vantagens; calibrar via Célula 5.2

def softmax_com_temperatura(scores, temperatura):
    scores = np.array(scores)
    scores_escalados = scores / temperatura
    scores_escalados -= np.max(scores_escalados)  # estabilidade numérica
    exp_scores = np.exp(scores_escalados)
    return exp_scores / np.sum(exp_scores)

# --- DIAGNÓSTICO TEMPORÁRIO: investigar a causa da entropia máxima generalizada ---
import collections

print("🔍 Shape da matriz de probabilidades:", probs_matrix.shape)
print(f"   (esperado: ({len(docs_para_classificar)}, 4) — se o número de colunas for maior que 4, "
      f"o BERTopic descobriu tópicos extras além dos 4 eixos zero-shot)")

print("\n🔍 Distribuição dos tópicos atribuídos pelo BERTopic (variável 'topics', hoje descartada):")
print(collections.Counter(topics))
print("   (se aparecerem valores diferentes de 0, 1, 2, 3 e -1, confirma tópicos extras descobertos)")

print("\n🔍 Informação geral dos tópicos do modelo:")
print(model_topic.get_topic_info())

print("\n🔍 Amostra das probabilidades CORRIGIDAS (mapeadas por topico_id) para os 5 primeiros versículos:")
for i in range(min(5, len(probs_matrix))):
    vals = {tid: float(probs_matrix[i][coluna_por_topico_id.get(tid, 0)]) for tid in topico_ids_db}
    print(f"  Verso {df_input.iloc[i]['verso_id']}: {vals} (soma: {sum(vals.values()):.3f})")

# 4. Processamento de Decisões Baseado em Concorrência Existencial e Incerteza
rows_verso_topico = []
rows_probabilidades = []

# Constantes matemáticas para Teoria da Informação (4 eixos = ln(4) de entropia máxima teórica)
ENTROPIA_MAXIMA = np.log(4)
# --- 4. PRÉ-CÁLCULO DINÂMICO DO LIMIAR ESTATÍSTICO (AÇÃO 3) ---
# Em vez de um valor hardcoded, coletamos todas as margens potenciais do corpus atual
# para derivar o limiar com base na dispersão real dos dados (ex: percentil 60).

margens_corpus_temp = []

for i in range(len(probs_matrix)):
    # Extrai probabilidades lineares normalizadas ou brutas para o cálculo prévio
    p_ex = float(probs_matrix[i][coluna_por_topico_id.get(0, 0)]) if 0 in coluna_por_topico_id else 0.0
    p_tr = float(probs_matrix[i][coluna_por_topico_id.get(1, 0)]) if 1 in coluna_por_topico_id else 0.0
    p_va = float(probs_matrix[i][coluna_por_topico_id.get(2, 0)]) if 2 in coluna_por_topico_id else 0.0
    p_na = float(probs_matrix[i][coluna_por_topico_id.get(3, 0)]) if 3 in coluna_por_topico_id else 0.0

    existenciais = [p_ex, p_tr, p_va]
    best_ex_score = max(existenciais) if existenciais else 0.0

    # Margem preliminar em relação ao normativo
    margem_temp = best_ex_score - p_na
    margens_corpus_temp.append(margem_temp)

# Derivação dinâmica: utiliza o Percentil 60 (P60) das margens do próprio livro como limiar de corte natural
PERCENTIL_ALVO = 60
LIMIAR_MARGEM_DINAMICO = float(np.percentile(margens_corpus_temp, PERCENTIL_ALVO))

print(f"📊 Limiar Estatístico Dinâmico Calculado (P{PERCENTIL_ALVO} do Corpus): {LIMIAR_MARGEM_DINAMICO:.4f}")

for i, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Avaliando", unit="v"):
    verso_id = int(row['verso_id'])

    prob_dict = {}
    for db_top_id in topico_ids_db:
        col_idx = coluna_por_topico_id.get(db_top_id)
        p_val = float(probs_matrix[i][col_idx]) if col_idx is not None and i < len(probs_matrix) else 0.0
        prob_dict[db_top_id] = p_val

        rows_probabilidades.append(
            (verso_id, execucao_id, int(db_top_id), p_val)
        )

    p_ex, p_tr, p_va, p_na = prob_dict.get(0, 0.0), prob_dict.get(1, 0.0), prob_dict.get(2, 0.0), prob_dict.get(3, 0.0)

    if row['texto_limpo'].startswith('RUIDO'):
        decisao_id = 3
        status = row['texto_limpo']
        best_score, margem, entropia, gap = 1.0, 1.0, 0.0, 1.0
    else:
        # Distribuição calibrada via softmax com temperatura para entropia
        probs_array = softmax_com_temperatura([p_ex, p_tr, p_va, p_na], TEMPERATURA_SOFTMAX)
        entropia = -np.sum(probs_array * np.log(probs_array + 1e-9))

        # Competição Intra-Existencial vs Normativo
        existenciais_dict = {0: p_ex, 1: p_tr, 2: p_va}
        existenciais_ordenados = sorted(existenciais_dict.items(), key=lambda x: x[1], reverse=True)

        best_existencial_id, best_existencial_score = existenciais_ordenados[0]
        segundo_existencial_id, segundo_existencial_score = existenciais_ordenados[1]

        # Gaps analíticos
        gap_existencial_interno = best_existencial_score - segundo_existencial_score
        margem_vs_normativo = best_existencial_score - p_na

        # O global gap (diferença entre o 1º e o 2º geral de todos os 4)
        todas_probs_ordenadas = sorted([p_ex, p_tr, p_va, p_na], reverse=True)
        gap = todas_probs_ordenadas[0] - todas_probs_ordenadas[1]

        # Regra de Decisão Refinada:
        # 1. Se o eixo Normativo/Narrativo (3) lidera de forma absoluta sobre todos os existenciais:
        if p_na > best_existencial_score:
            decisao_id = 3
            status = "Classificação Direta (Narrativo/Normativo Dominante)"
            best_score = p_na
            margem = p_na - best_existencial_score

        # 2. Se um eixo existencial lidera e passa pelo limiar de margem em relação ao normativo:
        elif margem_vs_normativo >= LIMIAR_MARGEM_DINAMICO:
            decisao_id = best_existencial_id
            status = f"Classificação por Margem (Dinâmica P{PERCENTIL_ALVO})"
            best_score = best_existencial_score
            margem = margem_vs_normativo

        # 3. Caso contrário, entra em fallback por margem insuficiente ou ambiguidade existencial
        else:
            decisao_id = 3
            status = "Fallback por Margem Insuficiente"
            best_score = best_existencial_score
            margem = margem_vs_normativo

    rows_verso_topico.append(
        (verso_id, execucao_id, int(decisao_id), status, float(entropia), float(gap), float(margem))
    )

# 5. Persistência Atômica no SQLite (Idempotente)
print("💾 Persistindo tabelas estruturadas (Suporte a reexecução via Upsert)...")

sql_verso_topico = """
    INSERT OR REPLACE INTO verso_topico (
        verso_id, execucao_id, topico_id, status_decisao,
        entropia, gap_confianca, margem_dominancia
    ) VALUES (?, ?, ?, ?, ?, ?, ?)
"""
cursor.executemany(sql_verso_topico, rows_verso_topico)

sql_probabilidade = """
    INSERT OR REPLACE INTO verso_topico_probabilidade (
        verso_id, execucao_id, topico_id, probabilidade
    ) VALUES (?, ?, ?, ?)
"""
cursor.executemany(sql_probabilidade, rows_probabilidades)

conn.commit()
conn.close()
print(f"✨ Célula 5 concluída! Vinculada à Execução ID: {execucao_id} com limiar estatístico global.")

In [ ]:
# Célula 5.1: Classificação por Eixos Existenciais via RAG com LLM Local (Gemma 2 na GPU T4)
!pip install -q chromadb sentence-transformers torch transformers accelerate bitsandbytes huggingface_hub pandas

import sqlite3
import chromadb
import os
import shutil
import json
import time
import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
import gc
from transformers import pipeline
from huggingface_hub import login
from google.colab import userdata, drive
from tqdm.notebook import tqdm

# --- 0. AUTENTICAÇÃO NO HUGGING FACE (Gated Repository) ---
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("🔑 Autenticação no Hugging Face realizada com sucesso.")
except Exception as e:
    print("⚠️ Erro ao autenticar no Hugging Face. Certifique-se de configurar o 'HF_TOKEN' nos segredos do Colab.")
    raise e

# --- 1. MONTAGEM BLINDADA DO DRIVE E CONFIGURAÇÃO DE CAMINHOS ---
drive_path = '/content/drive'
if not os.path.exists(os.path.join(drive_path, 'MyDrive')):
    try:
        drive.mount(drive_path)
    except Exception:
        get_ipython().system('umount /content/drive')
        drive.mount(drive_path, force_remount=True)
else:
    print("📁 Google Drive já está montado.")

base_dir = "/content/drive/MyDrive/pln/hybrid-rag-poc/data"
chroma_drive_path = os.path.join(base_dir, "chroma_db_moody")
chroma_local_path = "/content/chroma_db_moody"
sqlite_db_path = os.path.join(base_dir, "base-dados.db")

# --- 2. COPIAR CHROMADB PARA O SSD LOCAL ---
if os.path.exists(chroma_drive_path):
    print("📥 Copiando ChromaDB do Google Drive para o SSD local do Colab...")
    if os.path.exists(chroma_local_path):
        shutil.rmtree(chroma_local_path)
    shutil.copytree(chroma_drive_path, chroma_local_path)
else:
    print("⚠️ Atenção: A pasta do ChromaDB não foi encontrada no Drive.")

# Conecta ao ChromaDB localmente
chroma_client = chromadb.PersistentClient(path=chroma_local_path)
collection_name = "comentario_moody_jo"

try:
    collection = chroma_client.get_collection(name=collection_name)
    print(f"🎉 Coleção '{collection_name}' conectada com sucesso no ambiente local! Total de vetores: {collection.count()}")
except Exception as e:
    print(f"⚠️ A coleção '{collection_name}' não foi encontrada localmente.")
    raise e

espaco_metrica = (collection.metadata or {}).get("hnsw:space", "l2")  # "l2" é o padrão do Chroma quando omitido
if espaco_metrica != "cosine":
    raise ValueError(
        f"❌ A coleção '{collection_name}' usa métrica '{espaco_metrica}', não 'cosine'. "
        f"O cálculo de 'melhor_score = 1 - distance' assume distância de cosseno — reingira "
        f"a coleção especificando metadata={{'hnsw:space': 'cosine'}} na criação."
    )
print(f"✅ Métrica de distância da coleção confirmada: {espaco_metrica}")

# Conecta ao SQLite existente no Drive
conn = sqlite3.connect(sqlite_db_path)
cursor = conn.cursor()

# --- 3. GOVERNANÇA: RECUPERAR EXECUÇÃO E ATUALIZAR MODELOS (RAG e LLM Local) ---
if 'execucao_id' not in globals():
    cursor.execute("SELECT id, framework_versao_id FROM execucao_pipeline ORDER BY id DESC LIMIT 1")
    row = cursor.fetchone()
    if row:
        execucao_id, framework_versao_id = row
        print(f"🔄 Execução ID {execucao_id} recuperada do banco.")
    else:
        raise ValueError("❌ Nenhuma execução encontrada. Rode a Célula 3.1 primeiro.")
else:
    cursor.execute("SELECT framework_versao_id FROM execucao_pipeline WHERE id = ?", (execucao_id,))
    framework_versao_id = cursor.fetchone()[0]
    print(f"▶️ Utilizando Execução ID em memória: {execucao_id}")

# --- LER O MODELO DE EMBEDDINGS DIRETO DA COLEÇÃO, NUNCA HARDCODAR ---
# A ingestão roda em um notebook separado — a única forma segura de garantir consistência
# é a própria coleção "informar" com qual modelo ela foi indexada, em vez de duplicar a
# string do nome do modelo em dois notebooks que não compartilham execução nenhuma.
metadados_colecao = collection.metadata or {}
NOME_MODELO_EMBEDDING_RAG = metadados_colecao.get("embedding_model")

if NOME_MODELO_EMBEDDING_RAG is None:
    raise ValueError(
        f"❌ A coleção '{collection_name}' não tem o metadado 'embedding_model' registrado. "
        f"Esta coleção foi indexada com uma versão antiga do notebook de ingestão — "
        f"reingira os chunks com a versão atualizada antes de continuar, ou defina "
        f"NOME_MODELO_EMBEDDING_RAG manualmente aqui, com o modelo que você sabe ter sido usado."
    )

print(f"📌 Modelo de embeddings do RAG (lido da própria coleção): {NOME_MODELO_EMBEDDING_RAG}")
...
print(f"🔄 Carregando SentenceTransformer para consultas RAG: {NOME_MODELO_EMBEDDING_RAG}")
embedding_model = SentenceTransformer(NOME_MODELO_EMBEDDING_RAG, device="cuda")

# Definimos o modelo local na GPU
MODELO_LLM_LOCAL = "google/gemma-2-2b-it"

# Atualiza no banco o uso do RAG e do LLM local
cursor.execute("""
    UPDATE execucao_pipeline
    SET modelo_embeddings_rag = ?,
        modelo_llm_arbitragem = ?
    WHERE id = ?
""", (NOME_MODELO_EMBEDDING, MODELO_LLM_LOCAL, execucao_id))
conn.commit()

# --- 4. CARREGAR DEFINIÇÕES DINÂMICAS DOS TÓPICOS (SSOT) PARA O PROMPT ---
df_topicos_prompt = pd.read_sql_query(
    "SELECT id, nome_curto, descricao_prompt_llm FROM topico WHERE framework_versao_id = ? ORDER BY id",
    conn, params=(framework_versao_id,)
)
eixos_prompt_str = "\n".join([
    f"{row['id']}: {row['nome_curto']} ({row['descricao_prompt_llm']})"
    for _, row in df_topicos_prompt.iterrows()
])

cursor.execute("SELECT nome FROM livro WHERE id = ?", (LIVRO_ID_ALVO,))
NOME_LIVRO_ALVO = cursor.fetchone()[0]

# --- 5. LIMPEZA DE VRAM E CARREGAR MODELOS NA GPU COM OTIMIZAÇÃO ---
# Força a limpeza do cache da GPU para liberar memória órfã de execuções anteriores
torch.cuda.empty_cache()
gc.collect()

def gerar_embedding(texto):
    return embedding_model.encode([texto], show_progress_bar=False)[0].tolist()

print(f"🚀 Carregando LLM Local ({MODELO_LLM_LOCAL}) com Quantização de 4 bits na GPU T4...")

from transformers import BitsAndBytesConfig

# Configuração de quantização para economizar VRAM drástica (evita OOM)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

llm_pipeline = pipeline(
    "text-generation",
    model=MODELO_LLM_LOCAL,
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto"
)
print("✅ LLM Local carregado com sucesso e otimizado na GPU!")

# --- 6. IDENTIFICAÇÃO DOS CASOS CRÍTICOS (FILTRAGEM CIRÚRGICA REVISADA) ---
# Em vez de pegar quase todo o livro, focamos nos versículos de ambiguidade e disputa real

# --- FILTRO TEMPORÁRIO: restringe aos versículos com cobertura conhecida de chunks ---
# Esta condição replica o MESMO critério estrutural usado em buscar_contexto_no_chroma_com_proveniencia().
# Está aqui apenas porque a lista de chunks ingerida no ChromaDB ainda é reduzida (não cobre Jó inteiro).
# REMOVER esta cláusula assim que o corpus de comentário Moody for ingerido por completo,
# e reverter para o comportamento original de "todo Fallback é candidato ao RAG".

query_criticos = """
    SELECT v.id as verso_id, v.numero_capitulo, v.numero_verso, v.texto,
           vt.topico_id, vt.entropia, vt.gap_confianca, vt.status_decisao
    FROM verso_topico vt
    JOIN verso v ON v.id = vt.verso_id
    WHERE v.livro_id = ?
      AND vt.status_decisao = 'Fallback por Margem Insuficiente'
      AND vt.execucao_id = ?
      AND EXISTS (SELECT 1
            FROM chunks_comentario cc
            WHERE ((v.numero_capitulo BETWEEN cc.capitulo_inicio AND cc.capitulo_fim AND
                cc.verso_inicio IS NULL AND cc.verso_fim IS NULL) OR
                (v.numero_capitulo BETWEEN cc.capitulo_inicio AND cc.capitulo_fim AND
                v.numero_verso between cc.verso_inicio and cc.verso_fim)))
"""
df_criticos = pd.read_sql_query(query_criticos, conn, params=(LIVRO_ID_ALVO, execucao_id))

# TRECHO PROVISÓRIO, DEVERÁ SER REMOVIDO QUANDO TODO O CORPUS ESTIVER CONTEMPLADO POR CHUNKS
cursor.execute("""
    SELECT COUNT(*) FROM verso_topico vt
    JOIN verso v ON v.id = vt.verso_id
    WHERE v.livro_id = ? AND vt.status_decisao = 'Fallback por Margem Insuficiente' AND vt.execucao_id = ?
""", (LIVRO_ID_ALVO, execucao_id))
total_fallback_real = cursor.fetchone()[0]

print(f"🔍 RAG ativado de forma cirúrgica! Encontrados {len(df_criticos)} versículos "
      f"de alta ambiguidade real na Execução {execucao_id}.")
print(f"⚠️ Do total de {total_fallback_real} versículos em Fallback nesta execução, "
      f"{total_fallback_real - len(df_criticos)} foram excluídos por falta de chunk de "
      f"comentário correspondente (cobertura de comentário ainda parcial).")

cursor.execute("""
    UPDATE execucao_pipeline
    SET observacoes = observacoes || ' [AVISO: RAG restrito a versículos com chunk de comentário disponível — cobertura parcial]'
    WHERE id = ?
""", (execucao_id,))
# FIM TRECHO PROVISÓRIO

# --- 7. FUNÇÃO DO RETRIEVER HÍBRIDO ---
def buscar_contexto_no_chroma_com_proveniencia(texto_verso, capitulo, verso):
    q_emb = gerar_embedding(texto_verso)

    filtro_estrutural = {
        "$or": [
            {
                "$and": [
                    {"capitulo_inicio": {"$lte": int(capitulo)}},
                    {"capitulo_fim": {"$gte": int(capitulo)}},
                    {"verso_inicio": {"$lte": int(verso)}},
                    {"verso_fim": {"$gte": int(verso)}}
                ]
            },
            {
                "$and": [
                    {"capitulo_inicio": {"$lte": int(capitulo)}},
                    {"capitulo_fim": {"$gte": int(capitulo)}},
                    {"verso_inicio": {"$eq": 0}},
                    {"verso_fim": {"$eq": 0}}
                ]
            }
        ]
    }

    try:
        resultados = collection.query(
            query_embeddings=[q_emb],
            n_results=2,
            where=filtro_estrutural,
            include=["documents", "metadatas", "distances"]
        )
    except Exception:
        resultados = collection.query(
            query_embeddings=[q_emb],
            n_results=2,
            include=["documents", "metadatas", "distances"]
        )

    docs = resultados['documents'][0] if resultados['documents'] else []
    metas = resultados['metadatas'][0] if resultados['metadatas'] else []
    ids = resultados['ids'][0] if resultados['ids'] else []
    distances = resultados['distances'][0] if resultados['distances'] else []

    contexto_formatado = ""
    for doc, meta in zip(docs, metas):
        cap_ini = meta.get('capitulo_inicio', 'N/A')
        v_ini = meta.get('verso_inicio', 'N/A')
        cap_fim = meta.get('capitulo_fim', 'N/A')
        v_fim = meta.get('verso_fim', 'N/A')
        pag = meta.get('pagina_origem', 'N/A')
        contexto_formatado += f"- [Referência: Cap. {cap_ini} (v.{v_ini}) até Cap. {cap_fim} (v.{v_fim}) | Página: {pag}]\n{doc}\n\n"

    melhor_chunk_id = ids[0] if ids else "N/A"
    melhor_score = float(1.0 - distances[0]) if distances else 0.0

    return contexto_formatado.strip(), melhor_chunk_id, melhor_score

# --- 8. FUNÇÃO DE ARBITRAGEM VIA LLM LOCAL COM PARSER DE JSON ROBUSTO ---
import re
import json

def chamar_llm_local_arbitragem(verso_texto, contexto_comentario):
    prompt = f"""<|im_start|>[Instrução]
Você é um auditor de PLN especializado em análise teológica e filosófica de textos bíblicos.
Analise o versículo abaixo do livro de {NOME_LIVRO_ALVO} à luz do comentário exegético recuperado e classifique-o ESTRITAMENTE em um dos eixos abaixo:

{eixos_prompt_str}

[CONTEXTO EXEGÉTICO DO COMENTÁRIO MOODY]:
{contexto_comentario}

[VERSÍCULO ALVO]:
"{verso_texto}"

Retorne APENAS um objeto JSON válido contendo exatamente estas chaves, sem textos adicionais ou explicações fora do JSON:
{{
  "topico_id": <inteiro correspondente ao eixo>,
  "justificativa": "<explicação curta fundamentada no comentário>",
  "status_decisao": "RAG_Resgate"
}}
<|im_end|>
<|im_start|>[Resposta JSON]
"""
    try:
        outputs = llm_pipeline(
            prompt,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
            return_full_text=False
        )
        texto_gerado = outputs[0]['generated_text'].strip()

        # 1. Tenta extrair o bloco JSON padrão
        match_json = re.search(r'\{.*\}', texto_gerado, re.DOTALL)

        if match_json:
            json_str = match_json.group(0)
            try:
                return json.loads(json_str)
            except json.JSONDecodeError:
                # Se o JSON falhou por estar truncado (ex: cortado no meio da justificativa)
                pass

        # 2. SE O JSON ESTIVER TRUNCADO: Recuperação determinística de emergência
        # Captura o topico_id independentemente do resto estar cortado
        match_id = re.search(r'"topico_id"\s*:\s*([0-3])', texto_gerado)
        if match_id:
            topico_id_recuperado = int(match_id.group(1))

            # Tenta capturar o pedaço da justificativa que foi gerado antes do corte
            match_just = re.search(r'"justificativa"\s*:\s*"([^"]*)', texto_gerado, re.DOTALL)
            justificativa_recuperada = match_just.group(1) if match_just else "Justificativa truncada pelo modelo local."
            if not justificativa_recuperada.endswith('"'):
                justificativa_recuperada += "..." # Indica visualmente que foi cortado

            return {
                "topico_id": topico_id_recuperado,
                "justificativa": justificativa_recuperada,
                "status_decisao": "RAG_Resgate"
            }

        print(f"\n⚠️ Falha total na extração do JSON: {texto_gerado[:120]}...")
        return None

    except Exception as e:
        print(f"\n❌ Erro crítico no parse local: {e} | Texto gerado: {texto_gerado[:100]}")
        return None

# --- 9. EXECUTAR LOOP DE ARBITRAGEM (Com Chaves Compostas e verso_auditoria_rag) ---
import numpy as np

novas_decisoes_contador = 0
descartes_por_baixa_confianca = 0

if len(df_criticos) > 0:

    # --- FASE 1: RECUPERAÇÃO DE CONTEXTO PARA TODOS OS CASOS CRÍTICOS ---
    print("📚 Recuperando contexto exegético para todos os versículos críticos...")
    resultados_recuperacao = []
    for _, row in tqdm(df_criticos.iterrows(), total=len(df_criticos), desc="Recuperando Contexto RAG"):
        contexto, chunk_id, score = buscar_contexto_no_chroma_com_proveniencia(
            row['texto'], row['numero_capitulo'], row['numero_verso']
        )
        resultados_recuperacao.append({
            'verso_id': int(row['verso_id']),
            'topico_id_atual': int(row['topico_id']),
            'texto': row['texto'],
            'contexto': contexto,
            'chunk_id': chunk_id,
            'score': score
        })

    # --- FASE 2: LIMIAR DINÂMICO DE CONFIANÇA DO RETRIEVER ---
    # Mesma filosofia da Célula 5: em vez de uma constante arbitrária (ex.: "score < 0.3"),
    # calculamos o limiar como um percentil da distribuição observada NESTE lote crítico —
    # descarta o quartil de recuperações menos confiáveis, calibrado ao corpus atual.
    scores_lote = [r['score'] for r in resultados_recuperacao]
    PERCENTIL_SCORE_MINIMO = 25
    LIMIAR_SCORE_SIMILARIDADE = float(np.percentile(scores_lote, PERCENTIL_SCORE_MINIMO))
    print(f"📊 Limiar de confiança do retriever (P{PERCENTIL_SCORE_MINIMO} do lote crítico): "
          f"{LIMIAR_SCORE_SIMILARIDADE:.4f}")

    # --- FASE 3: ARBITRAGEM VIA LLM APENAS PARA CONTEXTOS CONFIÁVEIS ---
    print("🚀 Executando arbitragem RAG local na GPU (apenas para contextos com confiança suficiente)...")
    for r in tqdm(resultados_recuperacao, desc="Arbitrando RAG Local"):

        if r['score'] < LIMIAR_SCORE_SIMILARIDADE:
            # Contexto recuperado não é confiável o suficiente — NÃO arbitra, mantém a decisão
            # original da Célula 5 intacta, mas registra a tentativa para fins de auditoria.
            cursor.execute("""
                INSERT OR REPLACE INTO verso_auditoria_rag (
                    verso_id, execucao_id, topico_id_anterior, topico_id_novo,
                    justificativa, chunk_id, score_similaridade
                ) VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (
                r['verso_id'], execucao_id, r['topico_id_atual'], r['topico_id_atual'],
                f"RAG não arbitrou: score de similaridade ({r['score']:.3f}) abaixo do limiar "
                f"dinâmico P{PERCENTIL_SCORE_MINIMO} ({LIMIAR_SCORE_SIMILARIDADE:.3f}). "
                f"Decisão original da Célula 5 mantida.",
                r['chunk_id'], r['score']
            ))
            descartes_por_baixa_confianca += 1
            continue

        resposta_json = chamar_llm_local_arbitragem(r['texto'], r['contexto'])

        if resposta_json and 'topico_id' in resposta_json:
            novo_topico_id = int(resposta_json['topico_id'])
            justificativa_texto = str(resposta_json['justificativa'])
            status = str(resposta_json['status_decisao'])

            cursor.execute("""
                UPDATE verso_topico
                SET topico_id = ?,
                    status_decisao = ?
                WHERE verso_id = ? AND execucao_id = ?
            """, (novo_topico_id, status, r['verso_id'], execucao_id))

            cursor.execute("""
                INSERT OR REPLACE INTO verso_auditoria_rag (
                    verso_id, execucao_id, topico_id_anterior, topico_id_novo,
                    justificativa, chunk_id, score_similaridade
                ) VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (
                r['verso_id'], execucao_id, r['topico_id_atual'], novo_topico_id,
                justificativa_texto, r['chunk_id'], r['score']
            ))

            novas_decisoes_contador += 1

    conn.commit()
    print(f"\n✨ {novas_decisoes_contador} versos reclassificados via LLM local | "
          f"{descartes_por_baixa_confianca} descartados por baixa confiança do retriever (decisão original mantida).")
else:
    print("Nenhum verso crítico encontrado para processar nesta execução.")

conn.close()

In [ ]:
# Célula 6: Análise de Sentimento Contextual e Cruzamento Existencial (Com Chave Composta e Rastreabilidade)
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
from tqdm.auto import tqdm

# 1. Configuração e Conexão com o Banco de Dados
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# --- GOVERNANÇA: RECUPERAR EXECUÇÃO VIGENTE ---
if 'execucao_id' not in globals():
    cursor.execute("SELECT id FROM execucao_pipeline ORDER BY id DESC LIMIT 1")
    row = cursor.fetchone()
    if row:
        execucao_id = row[0]
        print(f"🔄 Execução ID {execucao_id} recuperada do banco para análise de sentimentos.")
    else:
        raise ValueError("❌ Nenhuma execução encontrada. Rode a Célula 3.1 primeiro.")
else:
    print(f"▶️ Analisando sentimentos para a Execução ID: {execucao_id}")

# 2. Inicializar o Analisador de Sentimento (PT-BR baseado em BERTimbau)
print("🚀 Carregando modelo Transformer para Sentimento (PT-BR)...")
analyzer = create_analyzer(task="sentiment", lang="pt")

# 3. Busca do texto original e dos tópicos restritos à execução vigente
query_dados = """
    SELECT v.id as verso_id, v.texto, vt.topico_id
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
    WHERE v.livro_id = ? AND vt.execucao_id = ?
"""
df_input = pd.read_sql_query(query_dados, conn, params=(LIVRO_ID_ALVO, execucao_id))

textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()

# 4. Execução da análise em lotes
print(f"📊 Analisando carga emocional de {len(textos)} versículos...")
sentimentos_para_persistir = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

for i in tqdm(range(0, len(textos), batch_size), desc="Analisando Sentimentos", unit="lote"):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        sentimentos_para_persistir.append((
            int(ids_lote[idx]),
            int(execucao_id),
            str(p.output),
            int(mapa_num.get(p.output, 0)),
            float(p.probas.get('POS', 0)),
            float(p.probas.get('NEG', 0)),
            float(p.probas.get('NEU', 0))
        ))

# 5. Persistência Idempotente no SQLite (Suporte a Chave Composta)
print("💾 Persistindo resultados na tabela verso_sentimento (Upsert)...")
sql_sentimento = """
    INSERT OR REPLACE INTO verso_sentimento (
        verso_id, execucao_id, label, sentimento_num, score_pos, score_neg, score_neu
    ) VALUES (?, ?, ?, ?, ?, ?, ?)
"""
cursor.executemany(sql_sentimento, sentimentos_para_persistir)
conn.commit()
print("\n✅ Célula 6 concluída! Sentimentos processados e salvos com sucesso.")

# 6. RESULTADO FINAL: O DIAGNÓSTICO (PROBLEMA) VS. A CURA (ANTÍDOTO)
print(f"\n📈 RESUMO EXECUTIVO: PROBLEMÁTICA (CRISE) VS. ANTÍDOTO (CURA) [Execução ID: {execucao_id}]")

res_final = pd.read_sql_query("""
    SELECT
        t.antidoto_referencia as Eixo_Filosofico,
        COUNT(*) as Total_Versos,
        SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Antidotos_Cura,
        SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Problematica_Crise,
        ROUND(AVG(vs.sentimento_num), 3) as Polaridade_Media
    FROM verso_topico vt
    JOIN topico t ON vt.topico_id = t.id
    JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id AND vs.execucao_id = vt.execucao_id
    WHERE t.id != 3 AND vt.execucao_id = ?
    GROUP BY t.antidoto_referencia
    ORDER BY Polaridade_Media DESC
""", conn, params=(execucao_id,))

conn.close()

# Exibe a tabela formatada no Colab
display(res_final)